
# Notebook 1: Nelore Cattle Data Audit and Cleaning

## Description
This notebook examines and prepares the Nelore cattle supplementation dataset for agricultural research. We will inspect the dataset's structure, identify variables and data types, check for missing values and inconsistencies, and clean the data where necessary. The goal is to ensure the dataset is reliable and suitable for analyzing associations between cattle growth, supplementation behaviour, pasture vegetation, and weather conditions.


In [ ]:

# Load the dataset and inspect its basic structure.
import pandas as pd
from pathlib import Path

file_path = Path("/content/Complete_Dataset.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "Upload Complete_Dataset.csv to the Colab session first."
    )

df = pd.read_csv(file_path, sep=None, engine="python")

print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())


Dataset dimensions: (390, 72)

Column names:
['START_DATE', 'FINAL_DATE', 'PERIOD', 'AMOUNT_DAYS', 'ANIMAL', 'START_WEIGHT', 'FINAL_WEIGHT', 'ADG', 'ADS', 'SUP_TIME', 'SUP_TOTAL', 'TOTAL_TIME', 'TOTAL_ATTENDANCE', 'TOTAL_DAYS', 'TOTAL_TIME (6-12h)', 'TOTAL_ATTENDANCE (6-12h)', 'TOTAL_DAYS (6-12h)', 'TOTAL_TIME (12-18h)', 'TOTAL_ATTENDANCE (12-18h)', 'TOTAL_DAYS (12-18h)', 'TOTAL_TIME (18-00h)', 'TOTAL_ATTENDANCE (18-00h)', 'TOTAL_DAYS (18-00h)', 'TOTAL_TIME (6-8h)', 'TOTAL_ATTENDANCE (6-8h)', 'TOTAL_DAYS (6-8h)', 'TOTAL_TIME (12-14h)', 'TOTAL_ATTENDANCE (12-14h)', 'TOTAL_DAYS (12-14h)', 'TOTAL_TIME (18-20h)', 'TOTAL_ATTENDANCE (18-20h)', 'TOTAL_DAYS (18-20h)', 'TOTAL_TIME_SPRING (09/23-12/21)', 'TOTAL_ATTENDANCE_SPRING (09/23-12/21)', 'TOTAL_DAYS_SPRING (09/23-12/21)', 'TOTAL_TIME_SUMMER (12/22-03/20)', 'TOTAL_ATTENDANCE_SUMMER (12/22-03/20)', 'TOTAL_DAYS_SUMMER (12/22-03/20)', 'TOTAL_TIME_AUTUMN (03/23-06/21)', 'TOTAL_ATTENDANCE_AUTUMN (03/23-06/21)', 'TOTAL_DAYS_AUTUMN (03/23-06/21)'

,START_DATE,FINAL_DATE,PERIOD,AMOUNT_DAYS,ANIMAL,START_WEIGHT,FINAL_WEIGHT,ADG,ADS,SUP_TIME,...,GEMI_INDEX,SAVI_INDEX,TEMP_AVG,RAIN_SUM,RAIN_AVG,RAD_SOL_SUM,RAD_SOL_AVG,EVAPOT,HUM_REL_AVG,PRES_ATM_AVG
0,2022-12-23,2023-01-20,1,28,1,300,310,"0,357",200.0,"6, 12, 18",...,-2.453009e+06,0.33659,25.343604,214.400003,7.393104,630.54,21.742759,137.928724,72.588691,963.151031
1,2022-12-23,2023-01-20,1,28,2,180,193,"0,464",200.0,"6, 12, 18",...,-2.453009e+06,0.33659,25.343604,214.400003,7.393104,630.54,21.742759,137.928724,72.588691,963.151031
2,2022-12-23,2023-01-20,1,28,3,250,271,"0,750",200.0,"6, 12, 18",...,-2.453009e+06,0.33659,25.343604,214.400003,7.393104,630.54,21.742759,137.928724,72.588691,963.151031
3,2022-12-23,2023-01-20,1,28,4,226,250,"0,857",200.0,"6, 12, 18",...,-2.453009e+06,0.33659,25.343604,214.400003,7.393104,630.54,21.742759,137.928724,72.588691,963.151031
4,2022-12-23,2023-01-20,1,28,5,173,196,"0,821",200.0,"6, 12, 18",...,-2.453009e+06,0.33659,25.343604,214.400003,7.393104,630.54,21.742759,137.928724,72.588691,963.151031


In [ ]:

# Check data types, missing values, and duplicate records.

print("DATA TYPES")
display(df.dtypes.to_frame(name="Data Type"))

print("\nMISSING VALUES")
missing = df.isnull().sum().to_frame(name="Missing Count")
missing["Missing Percentage (%)"] = (
    df.isnull().sum() / len(df) * 100
).round(2)
display(missing[missing["Missing Count"] > 0])

if df.isnull().sum().sum() == 0:
    print("No missing values detected.")

print("\nDUPLICATE ROWS")
print("Number of completely duplicated rows:", df.duplicated().sum())

print("\nUNIQUE ANIMALS AND PERIODS")
print("Unique animals:", df["ANIMAL"].nunique())
print("Unique periods:", df["PERIOD"].nunique())


DATA TYPES


,Data Type
START_DATE,object
FINAL_DATE,object
PERIOD,int64
AMOUNT_DAYS,int64
ANIMAL,int64
...,...
RAD_SOL_SUM,float64
RAD_SOL_AVG,float64
EVAPOT,float64
HUM_REL_AVG,float64



MISSING VALUES


,Missing Count,Missing Percentage (%)


No missing values detected.

DUPLICATE ROWS
Number of completely duplicated rows: 0

UNIQUE ANIMALS AND PERIODS
Unique animals: 26
Unique periods: 15



### Description
This step checks the quality and consistency of variables relevant to the research objectives. It examines observation dates, cattle weights, average daily weight gain (ADG), supplementation, feeder behaviour, vegetation indices, and weather conditions. The results will help identify potential data inconsistencies before cleaning the dataset and conducting statistical analyses.


In [ ]:

# Inspect key variables, date ranges, and weight-gain consistency.

# Convert dates temporarily for validation without changing the original columns.
start_dates = pd.to_datetime(df["START_DATE"], errors="coerce")
final_dates = pd.to_datetime(df["FINAL_DATE"], errors="coerce")

print("DATE CHECK")
print("Invalid start dates:", start_dates.isna().sum())
print("Invalid final dates:", final_dates.isna().sum())
print("Start date range:", start_dates.min(), "to", start_dates.max())
print("Final date range:", final_dates.min(), "to", final_dates.max())
print("Periods with final date before start date:",
      (final_dates < start_dates).sum())

# Inspect selected numeric columns.
print("\nKEY VARIABLE SUMMARIES")
key_columns = [
    "AMOUNT_DAYS", "START_WEIGHT", "FINAL_WEIGHT",
    "ADG", "ADS", "SUP_TOTAL", "TOTAL_TIME",
    "TOTAL_ATTENDANCE", "TOTAL_DAYS",
    "NDVI_INDEX", "NDWI_INDEX", "EVI_INDEX",
    "SAVI_INDEX", "TEMP_AVG", "RAIN_SUM"
]
display(df[key_columns].describe().T)

# Check whether recorded weight change matches ADG × duration.
adg_numeric = pd.to_numeric(
    df["ADG"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)
weight_change = df["FINAL_WEIGHT"] - df["START_WEIGHT"]
calculated_adg = weight_change / df["AMOUNT_DAYS"]

print("\nADG CONSISTENCY CHECK")
print("Rows with ADG that could not be converted:", adg_numeric.isna().sum())
print("Rows where recorded ADG differs from calculated ADG by more than 0.05 kg/day:",
      ((adg_numeric - calculated_adg).abs() > 0.05).sum())


DATE CHECK
Invalid start dates: 0
Invalid final dates: 0
Start date range: 2022-12-23 00:00:00 to 2023-10-04 00:00:00
Final date range: 2023-01-20 00:00:00 to 2023-10-26 00:00:00
Periods with final date before start date: 0

KEY VARIABLE SUMMARIES


,count,mean,std,min,25%,50%,75%,max
AMOUNT_DAYS,390.0,20.466667,4.339413,13.000000,17.000000,19.000000,24.000000,28.000000
START_WEIGHT,390.0,313.176923,58.447938,147.000000,274.250000,322.000000,357.000000,430.000000
FINAL_WEIGHT,390.0,324.956410,54.221536,156.000000,285.000000,330.000000,364.000000,447.000000
ADS,390.0,400.294500,165.204333,200.000000,200.000000,372.727273,600.000000,600.000000
SUP_TOTAL,390.0,10407.656997,4295.312646,5200.000000,5200.000000,9690.909091,15600.000000,15600.000000
TOTAL_TIME,390.0,1950.956410,3400.881169,0.000000,0.000000,535.000000,2463.750000,28779.000000
TOTAL_ATTENDANCE,390.0,19.620513,29.459215,0.000000,0.000000,10.000000,26.000000,202.000000
TOTAL_DAYS,390.0,4.633333,5.097145,0.000000,0.000000,3.000000,8.000000,18.000000
NDVI_INDEX,390.0,0.265017,0.085170,0.087370,0.194344,0.274443,0.333160,0.427202
NDWI_INDEX,390.0,0.084484,0.076386,-0.028299,-0.011722,0.107985,0.152259,0.191787



ADG CONSISTENCY CHECK
Rows with ADG that could not be converted: 0
Rows where recorded ADG differs from calculated ADG by more than 0.05 kg/day: 54



### Description
This step investigates inconsistencies in average daily weight gain (ADG) and checks supplementation and feeder-behaviour variables for unusual values. The aim is to distinguish genuine animal-level variation from possible data-recording, calculation, or interpretation issues before selecting variables for statistical analysis.


In [ ]:

# Compare recorded ADG with ADG calculated from weight change and duration.

adg_check = df[
    ["ANIMAL", "PERIOD", "AMOUNT_DAYS",
     "START_WEIGHT", "FINAL_WEIGHT", "ADG"]
].copy()

adg_check["ADG_recorded"] = pd.to_numeric(
    adg_check["ADG"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

adg_check["ADG_calculated"] = (
    (adg_check["FINAL_WEIGHT"] - adg_check["START_WEIGHT"])
    / adg_check["AMOUNT_DAYS"]
)

adg_check["ADG_difference"] = (
    adg_check["ADG_recorded"] - adg_check["ADG_calculated"]
)

adg_check["Difference_exceeds_0.05"] = (
    adg_check["ADG_difference"].abs() > 0.05
)

print("Number of records exceeding the screening threshold:",
      adg_check["Difference_exceeds_0.05"].sum())

print("\nExamples of discrepancies:")
display(
    adg_check.loc[
        adg_check["Difference_exceeds_0.05"]
    ].head(10).round(3)
)

print("\nSummary of absolute ADG differences:")
display(adg_check["ADG_difference"].abs().describe().to_frame(
    name="Absolute ADG difference"
))


Number of records exceeding the screening threshold: 54

Examples of discrepancies:


,ANIMAL,PERIOD,AMOUNT_DAYS,START_WEIGHT,FINAL_WEIGHT,ADG,ADG_recorded,ADG_calculated,ADG_difference,Difference_exceeds_0.05
34,9,2,27,252,291,"1,500",1.500,1.444,0.056,True
36,12,2,27,238,274,"1,385",1.385,1.333,0.052,True
38,14,2,27,202,242,"1,538",1.538,1.481,0.057,True
39,15,2,27,156,192,"1,385",1.385,1.333,0.052,True
42,18,2,27,235,272,"1,423",1.423,1.370,0.053,True
47,23,2,27,249,292,"1,654",1.654,1.593,0.061,True
51,27,2,27,244,281,"1,423",1.423,1.370,0.053,True
56,5,3,19,223,250,"1,500",1.500,1.421,0.079,True
72,22,3,19,298,318,"1,111",1.111,1.053,0.058,True
107,4,5,22,294,320,"1,238",1.238,1.182,0.056,True



Summary of absolute ADG differences:


,Absolute ADG difference
count,390.000000
mean,0.024072
std,0.023600
min,0.000000
25%,0.000429
50%,0.020289
75%,0.039281
max,0.132154



### Description
This step examines whether discrepancies between recorded and calculated average daily weight gain (ADG) occur consistently across observation periods or animals. Identifying these patterns will help determine whether the differences may reflect calculation methods, rounding, or other data inconsistencies.


In [ ]:

# Examine ADG discrepancies by observation period and animal.

adg_check["Absolute_difference"] = adg_check["ADG_difference"].abs()

print("ADG discrepancy summary by period:")
period_summary = adg_check.groupby("PERIOD").agg(
    Records=("ANIMAL", "size"),
    Mean_absolute_difference=("Absolute_difference", "mean"),
    Maximum_absolute_difference=("Absolute_difference", "max"),
    Records_over_threshold=("Difference_exceeds_0.05", "sum")
).round(4)

display(period_summary)

print("\nADG discrepancy summary by animal:")
animal_summary = adg_check.groupby("ANIMAL").agg(
    Records=("PERIOD", "size"),
    Mean_absolute_difference=("Absolute_difference", "mean"),
    Records_over_threshold=("Difference_exceeds_0.05", "sum")
).round(4)

display(animal_summary.head(10))


ADG discrepancy summary by period:


,Records,Mean_absolute_difference,Maximum_absolute_difference,Records_over_threshold
PERIOD,,,,
1,26,0.0002,0.0004,0
2,26,0.0445,0.0614,7
3,26,0.0331,0.0789,2
4,26,0.0002,0.0004,0
5,26,0.0371,0.0603,4
6,26,0.0485,0.0795,11
7,26,0.0582,0.1322,13
8,26,0.0270,0.0881,4
9,26,0.0421,0.0809,12



ADG discrepancy summary by animal:


,Records,Mean_absolute_difference,Records_over_threshold
ANIMAL,,,
1,15,0.0188,0
2,15,0.0192,1
3,15,0.0227,2
4,15,0.0220,2
5,15,0.0214,2
6,15,0.0227,1
7,15,0.0240,1
8,15,0.0239,1
9,15,0.0305,4



### Description
This step compares recorded average daily weight gain (ADG) with ADG calculated from weight change and observation duration across all study periods. The comparison will help identify when the two measures agree or diverge and guide the decision on which variable to use in subsequent analyses.


In [ ]:

# Compare recorded and calculated ADG across observation periods.

adg_period_comparison = adg_check.groupby("PERIOD").agg(
    Mean_recorded_ADG=("ADG_recorded", "mean"),
    Mean_calculated_ADG=("ADG_calculated", "mean"),
    Mean_difference=("ADG_difference", "mean"),
    Mean_absolute_difference=("Absolute_difference", "mean")
).round(4)

display(adg_period_comparison)

print("\nOverall comparison:")
print("Mean recorded ADG:",
      round(adg_check["ADG_recorded"].mean(), 4))
print("Mean calculated ADG:",
      round(adg_check["ADG_calculated"].mean(), 4))
print("Mean signed difference:",
      round(adg_check["ADG_difference"].mean(), 4))


,Mean_recorded_ADG,Mean_calculated_ADG,Mean_difference,Mean_absolute_difference
PERIOD,,,,
1,0.7500,0.7500,-0.0000,0.0002
2,1.2041,1.1595,0.0445,0.0445
3,0.6303,0.5972,0.0331,0.0331
4,0.9507,0.9505,0.0001,0.0002
5,0.8168,0.7797,0.0371,0.0371
6,0.7307,0.6851,0.0456,0.0485
7,0.7280,0.7840,-0.0561,0.0582
8,0.4722,0.4474,0.0249,0.0270
9,0.7118,0.6697,0.0421,0.0421



Overall comparison:
Mean recorded ADG: 0.5922
Mean calculated ADG: 0.5782
Mean signed difference: 0.014



### Description
This step investigates the relationship between recorded ADG and ADG calculated from weight change and observation duration. It checks whether discrepancies are systematic and whether the recorded values are sufficiently consistent for use as the primary cattle-growth outcome.


In [ ]:

# Measure agreement between recorded and calculated ADG.

print("Correlation between recorded and calculated ADG:")
print(
    round(
        adg_check["ADG_recorded"].corr(
            adg_check["ADG_calculated"]
        ),
        4
    )
)

print("\nMean signed difference by period:")
display(
    adg_check.groupby("PERIOD")["ADG_difference"]
    .mean()
    .round(4)
    .to_frame(name="Mean ADG difference")
)

print("\nDifference distribution:")
display(
    adg_check["ADG_difference"]
    .describe()
    .round(4)
    .to_frame(name="Recorded minus calculated ADG")
)


Correlation between recorded and calculated ADG:
0.9981

Mean signed difference by period:


,Mean ADG difference
PERIOD,
1,-0.0000
2,0.0445
3,0.0331
4,0.0001
5,0.0371
6,0.0456
7,-0.0561
8,0.0249
9,0.0421



Difference distribution:


,Recorded minus calculated ADG
count,390.0000
mean,0.0140
std,0.0307
min,-0.1322
25%,-0.0001
50%,0.0122
75%,0.0339
max,0.0881


In [ ]:
# Description: Summarize vegetation-index values to identify unusual ranges.

vegetation_columns = [
    "NDVI_INDEX",
    "NDWI_INDEX",
    "EVI_INDEX",
    "LAI_INDEX",
    "DVI_INDEX",
    "GCI_INDEX",
    "GEMI_INDEX",
    "SAVI_INDEX"
]

vegetation_summary = df[vegetation_columns].describe().T

display(vegetation_summary)

,count,mean,std,min,25%,50%,75%,max
NDVI_INDEX,390.0,2.650171e-01,8.516953e-02,8.737003e-02,1.943437e-01,2.744434e-01,3.331599e-01,4.272022e-01
NDWI_INDEX,390.0,8.448409e-02,7.638590e-02,-2.829924e-02,-1.172202e-02,1.079845e-01,1.522587e-01,1.917873e-01
EVI_INDEX,390.0,2.037947e-01,6.377916e-02,8.163885e-02,1.430952e-01,2.214792e-01,2.597741e-01,3.241821e-01
LAI_INDEX,390.0,6.193293e-01,2.307530e-01,1.773694e-01,3.997183e-01,6.833118e-01,8.218627e-01,1.054891e+00
DVI_INDEX,390.0,1.517863e+03,4.127932e+02,8.683554e+02,1.060914e+03,1.678347e+03,1.839966e+03,2.256369e+03
GCI_INDEX,390.0,7.836515e-01,2.502610e-01,1.546413e-01,6.368832e-01,7.762290e-01,9.942943e-01,1.236244e+00
GEMI_INDEX,390.0,-2.689124e+06,1.272679e+06,-5.091444e+06,-3.766135e+06,-3.019105e+06,-1.239008e+06,-1.069126e+06
SAVI_INDEX,390.0,3.974900e-01,1.277415e-01,1.310478e-01,2.914884e-01,4.116339e-01,4.996961e-01,6.407426e-01


The vegetation indices have no missing values. However, GEMI_INDEX contains unusually large negative values that require investigation. We will preserve the original values until their calculation method is verified.

We will inspect the highest and lowest GEMI_INDEX values alongside other vegetation indices and available satellite bands to investigate the unusual values.

In [ ]:
#  Inspect extreme GEMI values and related variables.

related_columns = [
    "ANIMAL", "PERIOD", "START_DATE", "FINAL_DATE",
    "GEMI_INDEX", "NDVI_INDEX", "NDWI_INDEX", "EVI_INDEX",
    "DVI_INDEX", "SAVI_INDEX"
]

band_columns = [col for col in df.columns if col.startswith("BAND")]
inspection_columns = related_columns + band_columns

print("Lowest 10 GEMI values:")
display(df.nsmallest(10, "GEMI_INDEX")[inspection_columns])

print("\nHighest 10 GEMI values:")
display(df.nlargest(10, "GEMI_INDEX")[inspection_columns])

Lowest 10 GEMI values:


,ANIMAL,PERIOD,START_DATE,FINAL_DATE,GEMI_INDEX,NDVI_INDEX,NDWI_INDEX,EVI_INDEX,DVI_INDEX,SAVI_INDEX,...,BAND3,BAND4,BAND5,BAND6,BAND7,BAND8,BAND8A,BAND9,BAND11,BAND12
130,1,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
131,2,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
132,3,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
133,4,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
134,5,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
135,6,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
136,7,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
137,8,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
138,9,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
139,11,6,2023-04-12,2023-04-28,-5.091444e+06,0.427202,0.157248,0.324182,2256.369393,0.640743,...,1685.201516,1512.369821,2294.256194,3610.94009,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531



Highest 10 GEMI values:


,ANIMAL,PERIOD,START_DATE,FINAL_DATE,GEMI_INDEX,NDVI_INDEX,NDWI_INDEX,EVI_INDEX,DVI_INDEX,SAVI_INDEX,...,BAND3,BAND4,BAND5,BAND6,BAND7,BAND8,BAND8A,BAND9,BAND11,BAND12
156,1,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
157,2,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
158,3,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
159,4,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
160,5,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
161,6,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
162,7,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
163,8,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
164,9,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
165,11,7,2023-04-28,2023-05-11,-1.069126e+06,0.08737,0.191787,0.081639,868.355404,0.131048,...,6178.78186,5842.49523,6467.192653,6727.73302,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953


The GEMI_INDEX values are extremely large and negative. The highest-value records show identical satellite measurements across multiple animals in Period 7, suggesting that vegetation data may be shared by period rather than measured separately for each animal. We need to investigate the GEMI calculation before making corrections.

We will inspect the GEMI values and available satellite-band columns to investigate the unusual values and identify possible calculation issues.

In [ ]:
# Description: Check GEMI values and related satellite measurements.

vegetation_columns = [
    "GEMI_INDEX", "NDVI_INDEX", "NDWI_INDEX", "EVI_INDEX",
    "DVI_INDEX", "SAVI_INDEX"
]

band_columns = [col for col in df.columns if col.startswith("BAND")]
inspection_columns = [
    col for col in [
        "ANIMAL", "PERIOD", "START_DATE", "FINAL_DATE"
    ] + vegetation_columns + band_columns
    if col in df.columns
]

print("GEMI summary:")
display(df["GEMI_INDEX"].describe())

print("\nGEMI values by period:")
display(
    df.groupby("PERIOD")["GEMI_INDEX"]
    .agg(["min", "max", "mean", "nunique"])
)

print("\nFirst five records with satellite measurements:")
display(df[inspection_columns].head())

GEMI summary:


,GEMI_INDEX
count,3.900000e+02
mean,-2.689124e+06
std,1.272679e+06
min,-5.091444e+06
25%,-3.766135e+06
50%,-3.019105e+06
75%,-1.239008e+06
max,-1.069126e+06



GEMI values by period:


,min,max,mean,nunique
PERIOD,,,,
1,-2.453009e+06,-2.453009e+06,-2.453009e+06,1
2,-3.996686e+06,-3.996686e+06,-3.996686e+06,1
3,-3.766135e+06,-3.766135e+06,-3.766135e+06,1
4,-4.081182e+06,-4.081182e+06,-4.081182e+06,1
5,-3.419483e+06,-3.419483e+06,-3.419483e+06,1
6,-5.091444e+06,-5.091444e+06,-5.091444e+06,1
7,-1.069126e+06,-1.069126e+06,-1.069126e+06,1
8,-3.361637e+06,-3.361637e+06,-3.361637e+06,1
9,-3.387034e+06,-3.387034e+06,-3.387034e+06,1



First five records with satellite measurements:


,ANIMAL,PERIOD,START_DATE,FINAL_DATE,GEMI_INDEX,NDVI_INDEX,NDWI_INDEX,EVI_INDEX,DVI_INDEX,SAVI_INDEX,...,BAND3,BAND4,BAND5,BAND6,BAND7,BAND8,BAND8A,BAND9,BAND11,BAND12
0,1,1,2022-12-23,2023-01-20,-2.453009e+06,0.224413,0.107985,0.173654,1262.475011,0.33659,...,4535.561583,4394.798519,5137.820589,5704.243059,5813.095543,5657.27353,5810.145807,8309.536523,4558.229532,3744.336466
1,2,1,2022-12-23,2023-01-20,-2.453009e+06,0.224413,0.107985,0.173654,1262.475011,0.33659,...,4535.561583,4394.798519,5137.820589,5704.243059,5813.095543,5657.27353,5810.145807,8309.536523,4558.229532,3744.336466
2,3,1,2022-12-23,2023-01-20,-2.453009e+06,0.224413,0.107985,0.173654,1262.475011,0.33659,...,4535.561583,4394.798519,5137.820589,5704.243059,5813.095543,5657.27353,5810.145807,8309.536523,4558.229532,3744.336466
3,4,1,2022-12-23,2023-01-20,-2.453009e+06,0.224413,0.107985,0.173654,1262.475011,0.33659,...,4535.561583,4394.798519,5137.820589,5704.243059,5813.095543,5657.27353,5810.145807,8309.536523,4558.229532,3744.336466
4,5,1,2022-12-23,2023-01-20,-2.453009e+06,0.224413,0.107985,0.173654,1262.475011,0.33659,...,4535.561583,4394.798519,5137.820589,5704.243059,5813.095543,5657.27353,5810.145807,8309.536523,4558.229532,3744.336466


GEMI_INDEX has extremely large negative values and only one unique value per period, shared across all animals. This suggests a possible calculation or scaling issue that needs investigation. We will preserve the original values.

We will compare the GEMI_INDEX values with the available satellite bands to investigate whether the unusual values could result from the calculation formula or band scaling.

In [ ]:
# Description: Inspect satellite-band ranges and their relationship with GEMI.

band_columns = [col for col in df.columns if col.startswith("BAND")]

print("Satellite band summary:")
display(df[band_columns].describe().T)

print("\nSatellite bands for one record per period:")
display(
    df.drop_duplicates("PERIOD")[
        ["PERIOD", "GEMI_INDEX"] + band_columns
    ].sort_values("PERIOD")
)

Satellite band summary:


,count,mean,std,min,25%,50%,75%,max
BAND1,390.0,2565.557116,1515.680275,1232.552648,1484.436566,2218.174370,3426.862879,6946.719351
BAND2,390.0,2537.922289,1389.549580,1287.156984,1551.230315,2184.633063,3384.398291,6543.971949
BAND3,390.0,2721.831294,1215.946011,1685.201516,1886.986188,2370.928407,3442.290901,6178.781860
BAND4,390.0,2683.655763,1118.621957,1512.369821,1770.358536,2514.398120,3233.044639,5842.495230
BAND5,390.0,3287.919996,1146.259025,2286.189022,2428.980706,2951.458885,3938.046241,6467.192653
BAND6,390.0,4051.603739,1026.928730,3056.419393,3217.557774,3637.815392,4707.466254,6727.733020
BAND7,390.0,4255.892870,985.729014,3228.578983,3458.552328,3876.757155,4852.877830,6782.777018
BAND8,390.0,4201.519127,956.606425,3199.158650,3520.122597,3768.739214,4989.469137,6710.850634
BAND8A,390.0,4376.361034,906.202848,3423.702691,3649.600812,4033.386907,4911.663071,6700.811192
BAND9,390.0,5627.930791,2037.401331,3606.333618,4022.171081,5149.896882,7107.525203,10552.465898



Satellite bands for one record per period:


,PERIOD,GEMI_INDEX,BAND1,BAND2,BAND3,BAND4,BAND5,BAND6,BAND7,BAND8,BAND8A,BAND9,BAND11,BAND12
0,1,-2.453009e+06,4683.612701,4487.806635,4535.561583,4394.798519,5137.820589,5704.243059,5813.095543,5657.273530,5810.145807,8309.536523,4558.229532,3744.336466
26,2,-3.996686e+06,1518.631639,1551.230315,1886.986188,1770.358536,2428.980706,3637.815392,3952.223836,3762.899900,4044.537164,4040.593123,3130.334401,2382.256158
52,3,-3.766135e+06,1798.767407,1803.799872,2105.291435,1968.346326,2602.817599,3775.104336,4079.876869,3874.662431,4175.083832,4263.994981,3198.818347,2502.465720
78,4,-4.081182e+06,2784.383882,2716.673359,2926.125872,2715.015520,3423.687883,4481.987185,4743.856044,4539.601880,4849.471309,5984.297736,3691.387726,2849.046419
104,5,-3.419483e+06,3547.066382,3384.398291,3501.881931,3313.422298,4024.919180,4918.595045,5127.031269,4991.769500,5183.387100,8386.067322,3837.674584,2929.677659
130,6,-5.091444e+06,1232.552648,1287.156984,1685.201516,1512.369821,2294.256194,3610.940090,3876.757155,3768.739214,4033.386907,4022.171081,2937.074968,2054.077531
156,7,-1.069126e+06,6946.719351,6543.971949,6178.781860,5842.495230,6467.192653,6727.733020,6782.777018,6710.850634,6700.811192,10552.465898,4457.969671,4067.448953
182,8,-3.361637e+06,3426.862879,3421.308308,3442.290901,3233.044639,3938.046241,4707.466254,4852.877830,4989.469137,4911.663071,7107.525203,3240.860423,2716.545173
208,9,-3.387034e+06,1290.481952,1376.363627,1699.466254,1680.156557,2286.189022,3217.557774,3458.552328,3520.122597,3649.600812,3715.145237,3060.233234,2237.217108
234,10,-3.019105e+06,1267.658266,1379.309355,1706.888851,1754.696882,2337.844425,3186.130543,3409.371551,3490.487569,3601.627937,3638.459889,3208.548911,2364.506279


The satellite bands contain values around 1,200–10,600, while GEMI_INDEX is extremely negative. This suggests a possible issue with the GEMI calculation or scaling, but we need to verify the formula before drawing a conclusion. Original values remain unchanged.

We will compare the recorded vegetation indices with values recalculated from the available satellite bands where the required formulas can be verified. This will help identify whether GEMI_INDEX may have been calculated incorrectly.

In [ ]:
# Description: List satellite-band columns for verifying vegetation-index formulas.

band_columns = [col for col in df.columns if col.startswith("BAND")]

print("Available satellite bands:")
print(band_columns)

Available satellite bands:
['BAND1', 'BAND2', 'BAND3', 'BAND4', 'BAND5', 'BAND6', 'BAND7', 'BAND8', 'BAND8A', 'BAND9', 'BAND11', 'BAND12']


The dataset contains 12 satellite bands, including BAND4 and BAND8, which are commonly used in vegetation-index calculations. We need to verify the original GEMI formula and band scaling before recalculating or changing any values.

We will compare the recorded NDVI_INDEX and EVI_INDEX with values calculated from the corresponding satellite bands. This will help us assess whether the bands are scaled consistently and whether the recorded indices match standard formulas.

In [ ]:
# Description: Compare recorded NDVI and EVI with standard band-based calculations.

# NDVI uses near-infrared (BAND8) and red (BAND4).
df["NDVI_calculated"] = (
    (df["BAND8"] - df["BAND4"]) /
    (df["BAND8"] + df["BAND4"])
)

# EVI standard formula using NIR, red, and blue bands.
df["EVI_calculated"] = (
    2.5 * (df["BAND8"] - df["BAND4"]) /
    (df["BAND8"] + 6 * df["BAND4"] - 7.5 * df["BAND2"] + 1)
)

print("Recorded versus calculated NDVI and EVI:")
display(
    df[
        [
            "PERIOD", "NDVI_INDEX", "NDVI_calculated",
            "EVI_INDEX", "EVI_calculated"
        ]
    ].drop_duplicates("PERIOD").sort_values("PERIOD").round(4)
)

Recorded versus calculated NDVI and EVI:


,PERIOD,NDVI_INDEX,NDVI_calculated,EVI_INDEX,EVI_calculated
0,1,0.2244,0.1256,0.1737,-1.9345
26,2,0.3598,0.3601,0.2764,1.8102
52,3,0.3332,0.3263,0.2598,2.2092
78,4,0.2744,0.2515,0.2311,10.0110
104,5,0.2812,0.2021,0.2215,-8.2323
130,6,0.4272,0.4273,0.3242,1.7682
156,7,0.0874,0.0692,0.0816,-0.2969
182,8,0.3004,0.2136,0.2320,-3.4546
208,9,0.3540,0.3538,0.2621,1.4027
234,10,0.3311,0.3309,0.2453,1.1809


The calculated NDVI values closely match the recorded values in some periods but differ in others. The calculated EVI values are unusually large, suggesting that the standard EVI formula may not be compatible with the current band scaling. We will preserve the original indices and investigate the calculation methods before making changes.

We will calculate the GEMI index using its standard formula and compare it with the recorded GEMI_INDEX. This will help us investigate whether the extreme negative values result from the formula or input scaling.

In [ ]:
# Description: Compare recorded GEMI with the standard GEMI formula.

nir = df["BAND8"].astype(float)
red = df["BAND4"].astype(float)

eta = (
    2 * (nir**2 - red**2)
    + 1.5 * nir
    + 0.5 * red
) / (nir + red + 0.5)

df["GEMI_calculated"] = (
    eta * (1 - 0.25 * eta)
    - (red - 0.125) / (1 - red)
)

print("Recorded versus calculated GEMI:")
display(
    df[
        ["PERIOD", "GEMI_INDEX", "GEMI_calculated"]
    ]
    .drop_duplicates("PERIOD")
    .sort_values("PERIOD")
    .round(4)
)

Recorded versus calculated GEMI:


,PERIOD,GEMI_INDEX,GEMI_calculated
0,1,-2.453009e+06,-1.592500e+06
26,2,-3.996686e+06,-3.967868e+06
52,3,-3.766135e+06,-3.631822e+06
78,4,-4.081182e+06,-3.327060e+06
104,5,-3.419483e+06,-2.815000e+06
130,6,-5.091444e+06,-5.088463e+06
156,7,-1.069126e+06,-7.531410e+05
182,8,-3.361637e+06,-3.083081e+06
208,9,-3.387034e+06,-3.383308e+06
234,10,-3.019105e+06,-3.010945e+06


The standard GEMI calculation also produces extremely large negative values, so the problem may involve band scaling or formula implementation. The recorded and calculated values differ across periods. We will preserve the original data and investigate further before deciding whether GEMI can be used.

We will check whether the satellite bands are stored as scaled reflectance values and compare the recorded vegetation indices across periods. This will help us determine whether the index calculations may require scaled inputs.

In [ ]:
# Description: Compare vegetation indices and satellite-band values across periods.

period_columns = [
    "PERIOD", "BAND2", "BAND4", "BAND8",
    "NDVI_INDEX", "EVI_INDEX", "GEMI_INDEX",
    "SAVI_INDEX", "DVI_INDEX"
]

period_summary = (
    df[period_columns]
    .drop_duplicates("PERIOD")
    .sort_values("PERIOD")
)

display(period_summary.round(4))

,PERIOD,BAND2,BAND4,BAND8,NDVI_INDEX,EVI_INDEX,GEMI_INDEX,SAVI_INDEX,DVI_INDEX
0,1,4487.8066,4394.7985,5657.2735,0.2244,0.1737,-2.453009e+06,0.3366,1262.4750
26,2,1551.2303,1770.3585,3762.8999,0.3598,0.2764,-3.996686e+06,0.5397,1992.5414
52,3,1803.7999,1968.3463,3874.6624,0.3332,0.2598,-3.766135e+06,0.4997,1906.3161
78,4,2716.6734,2715.0155,4539.6019,0.2744,0.2311,-4.081182e+06,0.4116,1824.5864
104,5,3384.3983,3313.4223,4991.7695,0.2812,0.2215,-3.419483e+06,0.4218,1678.3472
130,6,1287.1570,1512.3698,3768.7392,0.4272,0.3242,-5.091444e+06,0.6407,2256.3694
156,7,6543.9719,5842.4952,6710.8506,0.0874,0.0816,-1.069126e+06,0.1310,868.3554
182,8,3421.3083,3233.0446,4989.4691,0.3004,0.2320,-3.361637e+06,0.4505,1756.4245
208,9,1376.3636,1680.1566,3520.1226,0.3540,0.2621,-3.387034e+06,0.5310,1839.9660
234,10,1379.3094,1754.6969,3490.4876,0.3311,0.2453,-3.019105e+06,0.4966,1735.7907


Vegetation indices vary across the 15 periods. GEMI_INDEX contains unusually large negative values and requires verification, so its original values will be preserved and flagged for review.

We will check the main cattle growth, supplementation, and environmental variables for missing or unusual values before preparing the dataset for analysis.

In [ ]:
# Description: Check ranges and unusual values in key research variables.

key_variables = [
    "START_WEIGHT", "FINAL_WEIGHT", "ADG", "ADS",
    "SUP_TIME", "SUP_TOTAL", "TOTAL_TIME",
    "TOTAL_ATTENDANCE", "TOTAL_DAYS",
    "TEMP_AVG", "RAIN_SUM", "HUM_REL_AVG"
]

available_variables = [col for col in key_variables if col in df.columns]

display(df[available_variables].describe().T.round(2))

print("\nNegative values:")
display((df[available_variables].select_dtypes(include="number") < 0).sum().to_frame("Count"))

,count,mean,std,min,25%,50%,75%,max
START_WEIGHT,390.0,313.18,58.45,147.00,274.25,322.00,357.00,430.00
FINAL_WEIGHT,390.0,324.96,54.22,156.00,285.00,330.00,364.00,447.00
ADS,390.0,400.29,165.20,200.00,200.00,372.73,600.00,600.00
SUP_TOTAL,390.0,10407.66,4295.31,5200.00,5200.00,9690.91,15600.00,15600.00
TOTAL_TIME,390.0,1950.96,3400.88,0.00,0.00,535.00,2463.75,28779.00
TOTAL_ATTENDANCE,390.0,19.62,29.46,0.00,0.00,10.00,26.00,202.00
TOTAL_DAYS,390.0,4.63,5.10,0.00,0.00,3.00,8.00,18.00
TEMP_AVG,390.0,24.04,2.48,18.79,22.51,24.43,25.11,28.86
RAIN_SUM,390.0,89.57,70.35,0.80,16.70,68.60,155.70,214.40
HUM_REL_AVG,390.0,68.19,10.64,51.42,57.65,72.53,78.69,83.56



Negative values:


,Count
START_WEIGHT,0
FINAL_WEIGHT,0
ADS,0
SUP_TOTAL,0
TOTAL_TIME,0
TOTAL_ATTENDANCE,0
TOTAL_DAYS,0
TEMP_AVG,0
RAIN_SUM,0
HUM_REL_AVG,0


In [ ]:
# Description: Prepare ADG for analysis without overwriting the original values.

df["ADG_recorded"] = pd.to_numeric(
    df["ADG"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Dataset shape:", df.shape)
print("Missing ADG values after conversion:", df["ADG_recorded"].isna().sum())
print("Animals:", df["ANIMAL"].nunique())
print("Periods:", df["PERIOD"].nunique())

display(
    df[["START_WEIGHT", "FINAL_WEIGHT", "AMOUNT_DAYS", "ADG_recorded"]]
    .describe()
    .round(3)
)

Dataset shape: (390, 76)
Missing ADG values after conversion: 0
Animals: 26
Periods: 15


,START_WEIGHT,FINAL_WEIGHT,AMOUNT_DAYS,ADG_recorded
count,390.000,390.000,390.000,390.000
mean,313.177,324.956,20.467,0.592
std,58.448,54.222,4.339,0.493
min,147.000,156.000,13.000,-1.417
25%,274.250,285.000,17.000,0.306
50%,322.000,330.000,19.000,0.611
75%,357.000,364.000,24.000,0.944
max,430.000,447.000,28.000,1.714


In [ ]:
# Description: Save the prepared dataset for the next notebook.

output_file = "nelore_cattle_prepared.csv"
df.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"Dataset shape: {df.shape}")

Saved: nelore_cattle_prepared.csv
Dataset shape: (390, 76)
